In [ ]:
# [MARKDOWN]
# # NB06.1 — Attribution Faithfulness Metrics (CPU-only)
#
# **Scientific Goal:**
# Standard binary mIoU at an arbitrary threshold (e.g., top-50% mass) systematically under-reports the localization quality of inherently focal attribution maps against loose bounding boxes. This supplementary notebook implements four metric families for True Positives (where a consensus GT box exists AND the model predicted the class):
#
# 1. **Precision (Mass-in-Box)** and **Recall (Binary Coverage)**: Precision asks "Of all attribution mass, what fraction falls inside GT?" Recall asks "What fraction of the GT box area is covered by the top-50% mask?".
# 2. **AUC-mIoU**: Threshold-agnostic mIoU across a full mass-fraction sweep (1% to 100%).
# 3. **Continuous correlation**: Point-Biserial and Spearman correlation between continuous IG intensity and the binary GT mask (avoids thresholding).
# 4. **Tolerance-margin mIoU**: GT boxes expanded by 5% image width (matching NB07 FP spatial analysis).
#
# Legacy mIoU (top50, no margin) is retained and cross-checked against existing results.



In [ ]:
from google.colab import drive
drive.mount('/content/drive')
GDRIVE_ROOT = '/content/drive/MyDrive/cxr_faithfulness'
exec(open(f'{GDRIVE_ROOT}/config/startup.py').read())



Mounted at /content/drive
⏳ Installing strictly pinned architecture packages onto Colab's native stack (~30s)...
✅ Packages ready! Using native modern PyTorch and NumPy.


In [ ]:
import csv
import warnings
import hashlib
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import scipy.stats as stats

try:
    from statsmodels.stats.multitest import multipletests
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels'])
    from statsmodels.stats.multitest import multipletests

FORBIDDEN = ('torch', 'captum', 'tensorflow')
for mod in FORBIDDEN:
    assert mod not in sys.modules, f"GPU package {mod} imported — this notebook must be CPU-only"



In [ ]:
ROOT = Path(GDRIVE_ROOT)
RESULTS_PATH = ROOT / 'results'
DATA_PATH = ROOT / 'data' / 'processed'
IG_PATH = ROOT / 'ig_maps'
ANN_PATH = DATA_PATH / 'consensus'
MASK_RES = 224

ALPHA = 0.05
BOOTSTRAP_N = 1000
MIN_TEST_N = 30
WILCOX_MIN_N = 15
SEED = 42
MODEL_ORDER = ['densenet121', 'convnextv2_tiny', 'swinb_lora']
GT_MARGIN_FRAC = 0.05   # 5% of 224px ≈ 11px — matches phase3_continuous_check.py
PRIMARY_CONSENSUS = '2of3'
SENS_CONSENSUS = '3of3'



In [ ]:
def stable_seed(text):
    return int(hashlib.md5(text.encode('utf-8')).hexdigest(), 16) % 100000

def bootstrap_ci(values, n_boot=BOOTSTRAP_N, ci=0.95, seed=None):
    vals = np.asarray(values, dtype=float)
    vals = vals[~np.isnan(vals)]
    if len(vals) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    n = len(vals)
    boots = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boots[i] = vals[idx].mean()
    lo_q = (1 - ci) / 2
    hi_q = 1 - lo_q
    return float(boots.mean()), float(np.quantile(boots, lo_q)), float(np.quantile(boots, hi_q))

def safe_round(x, nd=4):
    return float(np.round(x, nd)) if pd.notna(x) else np.nan



In [ ]:
def load_bbox_table(consensus='2of3'):
    path = ANN_PATH / f'consensus_boxes_{consensus}.csv'
    assert path.exists(), f'Missing: {path}'
    return pd.read_csv(path), path

def find_box_columns(df):
    colmap = {}
    for target, aliases in {
        'image_id':['image_id','id'],
        'target_class':['target_class','class','pathology','class_name'],
        'xmin':['xmin','x1','left', 'x_min'],
        'ymin':['ymin','y1','top', 'y_min'],
        'xmax':['xmax','x2','right', 'x_max'],
        'ymax':['ymax','y2','bottom', 'y_max']
    }.items():
        found = next((c for c in aliases if c in df.columns), None)
        if found is None:
            raise ValueError(f'Missing bbox column for {target}. Found {df.columns.tolist()}')
        colmap[target] = found
    return colmap

def resolve_path(p: str | Path) -> Path:
    p = Path(p)
    if p.exists():
        return p
    # strip to cxr_faithfulness/... suffix
    parts = str(p).replace('\\', '/').split('cxr_faithfulness/')
    if len(parts) > 1:
        candidate = ROOT / parts[-1]
        if candidate.exists():
            return candidate
    # try ig_maps/ basename fallback
    candidate = IG_PATH / p.name
    if candidate.exists():
        return candidate
    return p  # let caller handle missing



In [ ]:
def rasterize_boxes(boxes, shape=(MASK_RES, MASK_RES), margin_frac=0.0) -> np.ndarray:
    """boxes: list of (x1,y1,x2,y2) float. Returns uint8 {0,1} mask."""
    h, w = shape
    mask = np.zeros(shape, dtype=np.uint8)
    for x1, y1, x2, y2 in boxes:
        if margin_frac == 0.0:
            # Match NB06 compute_miou_file exactly (int truncate, not floor/ceil)
            xa = int(max(0, float(x1)))
            ya = int(max(0, float(y1)))
            xb = int(min(w, float(x2)))
            yb = int(min(h, float(y2)))
        else:
            mrg = margin_frac * w
            xa = int(max(0, np.floor(x1 - mrg)))
            ya = int(max(0, np.floor(y1 - mrg)))
            xb = int(min(w, np.ceil(x2 + mrg)))
            yb = int(min(h, np.ceil(y2 + mrg)))
        if xa < xb and ya < yb:
            mask[ya:yb, xa:xb] = 1
    return mask


def miou_from_masks(pred_mask: np.ndarray, gt_mask: np.ndarray) -> float:
    """Pixel IoU — same formula as NB06 compute_miou_file."""
    pred = pred_mask.astype(np.uint8)
    gt = gt_mask.astype(np.uint8)
    inter = np.logical_and(pred, gt).sum()
    uni = np.logical_or(pred, gt).sum()
    return float(inter / uni) if uni > 0 else 0.0


def top_mass_mask(attr_map: np.ndarray, mass: float = 0.5) -> np.ndarray:
    flat = attr_map.flatten().astype(np.float32)
    order = np.argsort(flat)[::-1]
    vals = flat[order]
    csum = np.cumsum(vals)
    total = csum[-1]

    if total <= 0:
        return np.zeros_like(attr_map, dtype=np.uint8)

    cutoff = int(np.searchsorted(csum, mass * total, side='left')) + 1
    keep_idx = order[:cutoff]

    mask = np.zeros_like(flat, dtype=np.uint8)
    mask[keep_idx] = 1
    return mask.reshape(attr_map.shape)



In [ ]:
import json
with open(ROOT / 'models/thresholds.json', 'r', encoding='utf-8') as f:
    th = json.load(f)

prob_lookup = {}
for m in MODEL_ORDER:
    try:
        with open(ROOT / f'results/{m}_test_image_probs.csv', 'r', encoding='utf-8') as f:
            for row in csv.DictReader(f):
                prob_lookup[(str(row['image_id']), m)] = {k: float(v) for k, v in row.items() if k != 'image_id'}
    except FileNotFoundError: pass

assert len(prob_lookup) > 0, "prob_lookup empty — check results/*_test_image_probs.csv paths"

patho_gt = pd.read_csv(DATA_PATH / 'splits/test_patho.csv')
patho_gt['image_id'] = patho_gt['image_id'].astype(str)
patho_gt = patho_gt.set_index('image_id')

def is_true_positive(image_id, model, target_class):
    image_id = str(image_id)
    target_class = str(target_class)
    thr = th.get(model, {}).get(target_class, 0.5)
    probs = prob_lookup.get((image_id, model), {})
    p = probs.get(target_class)
    if p is None or p < thr:
        return False
    if target_class not in patho_gt.columns:
        return False
    if image_id not in patho_gt.index:
        return False
    return float(patho_gt.loc[image_id, target_class]) == 1.0



In [ ]:
# PRE-FLIGHT VALIDATION
REQUIRED = [
    RESULTS_PATH / 'ig_manifest.csv',
    RESULTS_PATH / 'miou_results.csv',
    ANN_PATH / 'consensus_boxes_2of3.csv',
    ANN_PATH / 'consensus_boxes_3of3.csv',
    DATA_PATH / 'splits/test_patho.csv',
    ROOT / 'models/thresholds.json',
]
for p in REQUIRED:
    assert p.exists(), f"Missing required input: {p}"

for m in MODEL_ORDER:
    prob_file = RESULTS_PATH / f'{m}_test_image_probs.csv'
    assert prob_file.exists(), f"Missing {prob_file}"

manifest_check = pd.read_csv(RESULTS_PATH / 'ig_manifest.csv')
assert {'image_id','model','subset','target_class','ig_path','top50_path'}.issubset(manifest_check.columns)

patho_rows = manifest_check[manifest_check['subset'] == 'patho']
tp_count = sum(
    is_true_positive(str(r.image_id), str(r.model), str(r.target_class))
    for r in patho_rows.itertuples(index=False)
)
print(f"Manifest patho rows: {len(patho_rows)}")
print(f"TP-filtered rows (pre-box): {tp_count}")
assert tp_count > 100, f"TP count suspiciously low ({tp_count}) — check ID normalization / prob_lookup"



Manifest patho rows: 1383
TP-filtered rows (pre-box): 1030


In [15]:
def run_metrics_pipeline(consensus=PRIMARY_CONSENSUS):
    manifest_path = RESULTS_PATH / 'ig_manifest.csv'
    manifest = pd.read_csv(manifest_path)
    bbox_df, bbox_src = load_bbox_table(consensus)
    cols = find_box_columns(bbox_df)

    # Normalize keys
    bbox_df[cols['image_id']] = bbox_df[cols['image_id']].astype(str)
    bbox_grouped = bbox_df.groupby([cols['image_id'], cols['target_class']])

    rows = []
    sweep_rows = []
    MASS_FRACS = np.linspace(0.01, 1.00, 100)
    REPORT_FRACS = (0.30, 0.50, 0.70)

    n_patho = n_tp = n_has_box = n_has_ig = 0

    for r in tqdm(manifest.itertuples(index=False), desc=f'Processing {consensus}', total=len(manifest)):
        if r.subset != 'patho':
            continue
        n_patho += 1

        image_id = str(r.image_id)
        target_class = str(r.target_class)
        model = str(r.model)

        # TP Filter
        if not is_true_positive(image_id, model, target_class):
            continue
        n_tp += 1

        key = (image_id, target_class)
        if key not in bbox_grouped.groups:
            continue
        n_has_box += 1

        gt = bbox_grouped.get_group(key)
        if gt.empty:
            continue

        ig_path = resolve_path(r.ig_path)
        if not ig_path.exists():
            warnings.warn(f'Missing IG: {ig_path}')
            continue

        # FIX 3: Require top50_path for miou_top50
        top50_p = getattr(r, 'top50_path', None)
        if pd.isna(top50_p):
            warnings.warn(f"Missing top50_path for {image_id} {model} — skipping row")
            continue
        top50_path = resolve_path(top50_p)
        if not top50_path.exists():
            warnings.warn(f"Missing top50_path file: {top50_path} — skipping row")
            continue

        n_has_ig += 1

        pred_top50 = np.load(top50_path)
        if pred_top50.ndim == 3:
            pred_top50 = pred_top50.squeeze()
        pred_top50 = pred_top50.astype(np.uint8)

        boxes = [(float(g[cols['xmin']]), float(g[cols['ymin']]), float(g[cols['xmax']]), float(g[cols['ymax']])) for _, g in gt.iterrows()]

        ig_raw = np.load(ig_path).astype(np.float64)
        ig_raw = np.abs(ig_raw)
        if ig_raw.ndim == 3:
            ig_raw = ig_raw.sum(axis=0)

        ig_sum = ig_raw.sum()
        ig_norm = ig_raw / ig_sum if ig_sum > 1e-12 else np.zeros_like(ig_raw)

        gt_mask = rasterize_boxes(boxes, margin_frac=0.0)
        gt_mask_m5 = rasterize_boxes(boxes, margin_frac=GT_MARGIN_FRAC)

        B = gt_mask.astype(bool)
        B_mean = B.mean()
        if B_mean == 0:
            continue

        # A. Continuous metrics
        precision_mass = ig_norm[B].sum()
        enrichment_ratio = precision_mass / B_mean
        null_precision = B_mean
        delta_precision = precision_mass - null_precision

        # Load precomputed masks
        precomputed_masks = {}
        for mass_frac, p_attr in [(0.30, 'top30_path'), (0.50, 'top50_path'), (0.70, 'top70_path')]:
            p_val = getattr(r, p_attr, None)
            if pd.notna(p_val):
                p_res = resolve_path(p_val)
                if p_res.exists():
                    loaded = np.load(p_res)
                    if loaded.ndim == 3: loaded = loaded.squeeze()
                    precomputed_masks[mass_frac] = loaded.astype(bool)

        # B & C & D. Threshold sweep
        miou_vals = []
        miou_m5_vals = []
        bin_metrics = {}
        TARGET_SET = {0.30, 0.50, 0.70}

        for f in MASS_FRACS:
            f_round = round(f, 2)
            if f_round in precomputed_masks:
                pred_mask = precomputed_masks[f_round]
            else:
                pred_mask = top_mass_mask(ig_raw, f).astype(bool)

            pred_n = pred_mask.sum()

            # Regular
            inter = (pred_mask & B).sum()
            gt_n = B.sum()
            precision_bin = inter / pred_n if pred_n > 0 else 0.0
            recall_bin = inter / gt_n if gt_n > 0 else 0.0
            union = pred_n + gt_n - inter
            miou_bin = inter / union if union > 0 else 0.0
            miou_vals.append(miou_bin)

            # Margin
            inter_m5 = (pred_mask & gt_mask_m5.astype(bool)).sum()
            union_m5 = pred_n + gt_mask_m5.sum() - inter_m5
            miou_bin_m5 = inter_m5 / union_m5 if union_m5 > 0 else 0.0
            miou_m5_vals.append(miou_bin_m5)

            # Save targeted fracs
            if f_round in TARGET_SET:
                bin_metrics[float(f_round)] = {
                    'precision': precision_bin, 'recall': recall_bin, 'miou': miou_bin,
                    'miou_m5': miou_bin_m5, 'precision_m5': inter_m5/pred_n if pred_n > 0 else 0.0,
                    'recall_m5': inter_m5/gt_mask_m5.sum() if gt_mask_m5.sum() > 0 else 0.0
                }

            sweep_rows.append({
                'image_id': image_id, 'model': model, 'target_class': target_class,
                'mass_fraction': f_round, 'miou': miou_bin, 'miou_margin5': miou_bin_m5,
                'pred_area_px': pred_n, 'gt_area_px': gt_n, 'precision_bin': precision_bin, 'recall_bin': recall_bin
            })

        missing = [f for f in REPORT_FRACS if f not in bin_metrics]
        if missing:
            raise ValueError(f"Missing bin_metrics for {missing} — check MASS_FRACS / precomputed masks")

        # NB06-parity top50 metrics: same pred file + same GT rasterization as NB06
        pred_bool = pred_top50.astype(bool)
        gt_n = B.sum()
        pred_n = int(pred_bool.sum())
        inter50 = int((pred_bool & B).sum())
        miou_top50_nb06 = miou_from_masks(pred_top50, gt_mask)
        bin_metrics[0.50] = {
            'precision': inter50 / pred_n if pred_n > 0 else 0.0,
            'recall': inter50 / gt_n if gt_n > 0 else 0.0,
            'miou': miou_top50_nb06,
            'miou_m5': bin_metrics[0.50]['miou_m5'],
            'precision_m5': bin_metrics[0.50]['precision_m5'],
            'recall_m5': bin_metrics[0.50]['recall_m5'],
        }

        auc_miou = np.trapz(miou_vals, dx=0.01) / 0.99
        auc_miou_margin5 = np.trapz(miou_m5_vals, dx=0.01) / 0.99
        max_miou = max(miou_vals)
        max_miou_margin5 = max(miou_m5_vals)

        # E. Continuous correlation
        B_flat = gt_mask.ravel().astype(float)
        g_flat = ig_norm.ravel()
        if ig_sum > 1e-12:
            pb_r, pb_p = stats.pointbiserialr(B_flat, g_flat)
            sp_r, sp_p = stats.spearmanr(g_flat, B_flat)
        else:
            pb_r, pb_p, sp_r, sp_p = np.nan, np.nan, np.nan, np.nan

        # Compile row
        row_dict = {
            'image_id': image_id, 'model': model, 'target_class': target_class, 'subset': r.subset, 'consensus': consensus,
            'precision_mass': precision_mass, 'enrichment_ratio': enrichment_ratio, 'null_precision': null_precision, 'delta_precision': delta_precision,
            'precision_top30': bin_metrics[0.30]['precision'], 'recall_top30': bin_metrics[0.30]['recall'], 'miou_top30': bin_metrics[0.30]['miou'],
            'precision_top50': bin_metrics[0.50]['precision'], 'recall_top50': bin_metrics[0.50]['recall'], 'miou_top50': bin_metrics[0.50]['miou'],
            'precision_top70': bin_metrics[0.70]['precision'], 'recall_top70': bin_metrics[0.70]['recall'], 'miou_top70': bin_metrics[0.70]['miou'],
            'miou_top50_margin5': bin_metrics[0.50]['miou_m5'], 'precision_top50_margin5': bin_metrics[0.50]['precision_m5'], 'recall_top50_margin5': bin_metrics[0.50]['recall_m5'],
            'auc_miou': auc_miou, 'max_miou': max_miou, 'auc_miou_margin5': auc_miou_margin5, 'max_miou_margin5': max_miou_margin5,
            'pointbiserial_r': pb_r, 'pointbiserial_p': pb_p, 'spearman_r': sp_r, 'spearman_p': sp_p, 'ig_path': str(ig_path)
        }
        rows.append(row_dict)

    print(f"Filter funnel ({consensus}): patho={n_patho} → TP={n_tp} → has_box={n_has_box} → has_ig={n_has_ig} → output={len(rows)}")
    if len(rows) == 0:
        raise RuntimeError("No rows produced — TP filter or path resolution failed. Do not save empty CSVs.")

    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_PATH / f'attribution_metrics_{consensus}.csv', index=False)
    df_sweep = pd.DataFrame(sweep_rows)
    df_sweep.to_csv(RESULTS_PATH / f'attribution_threshold_sweep_{consensus}.csv', index=False)

    print(f"✅ attribution_metrics_{consensus}.csv: {len(df)} rows")
    return df



In [16]:
def aggregate_and_test(df, consensus):
    if df.empty:
        raise RuntimeError("Cannot aggregate empty dataframe")

    sens_path = RESULTS_PATH / 'retrospective_sensitivity.csv'
    if sens_path.exists():
        sens_df = pd.read_csv(sens_path)
        n_test_map = dict(zip(sens_df['pathology'], sens_df['n_test']))
    else:
        n_test_map = {}

    # 1. Aggregated paper table
    agg_rows = []
    for (model, patho), sub in df.groupby(['model', 'target_class']):
        n = len(sub)
        seed_base = stable_seed(f"{model}_{patho}")

        row_out = {'model': model, 'pathology': patho, 'n': n}
        row_out['flag_n30'] = '†' if n_test_map.get(patho, 999) < MIN_TEST_N else ''

        for met in ['precision_mass', 'recall_top50', 'delta_precision', 'precision_top50', 'miou_top50', 'auc_miou', 'auc_miou_margin5', 'pointbiserial_r', 'spearman_r']:
            mean_val, lo, hi = bootstrap_ci(sub[met].dropna().values, seed=seed_base)
            row_out[f'{met}_mean'] = safe_round(mean_val)
            row_out[f'{met}_ci_lo'] = safe_round(lo)
            row_out[f'{met}_ci_hi'] = safe_round(hi)

        agg_rows.append(row_out)

    agg_df = pd.DataFrame(agg_rows)
    agg_df.to_csv(RESULTS_PATH / f'attribution_faithfulness_{consensus}.csv', index=False)

    # 2. Model-level summary
    mod_rows = []
    for model, sub in df.groupby('model'):
        n = len(sub)
        seed_base = stable_seed(f"{model}_all")
        row_out = {'model': model, 'n': n}
        for met in ['precision_mass', 'recall_top50', 'delta_precision', 'miou_top50', 'auc_miou', 'pointbiserial_r']:
            mean_val, lo, hi = bootstrap_ci(sub[met].dropna().values, seed=seed_base)
            row_out[f'{met}_mean'] = safe_round(mean_val)
            row_out[f'{met}_ci_lo'] = safe_round(lo)
            row_out[f'{met}_ci_hi'] = safe_round(hi)
        mod_rows.append(row_out)

    mod_df = pd.DataFrame(mod_rows)
    mod_df.to_csv(RESULTS_PATH / f'attribution_summary_by_model_{consensus}.csv', index=False)

    # 3. Paired Wilcoxon
    test_metrics = ['precision_mass', 'recall_top50', 'auc_miou', 'pointbiserial_r']
    wilcox_rows = []

    for patho in sorted(df['target_class'].unique()):
        for i in range(len(MODEL_ORDER)):
            for j in range(i+1, len(MODEL_ORDER)):
                m1, m2 = MODEL_ORDER[i], MODEL_ORDER[j]
                sub1 = df[(df['model'] == m1) & (df['target_class'] == patho)][['image_id'] + test_metrics]
                sub2 = df[(df['model'] == m2) & (df['target_class'] == patho)][['image_id'] + test_metrics]

                df12 = sub1.merge(sub2, on='image_id', suffixes=('_1', '_2'))
                if len(df12) < WILCOX_MIN_N:
                    continue

                for met in test_metrics:
                    diffs = df12[f'{met}_1'] - df12[f'{met}_2']
                    if np.all(diffs == 0):
                        continue

                    w_stat, w_p = stats.wilcoxon(diffs)
                    wilcox_rows.append({
                        'pathology': patho, 'metric': met, 'm1': m1, 'm2': m2, 'n': len(df12),
                        'm1_mean': df12[f'{met}_1'].mean(), 'm2_mean': df12[f'{met}_2'].mean(),
                        'wilcox_stat': w_stat, 'wilcox_p_uncorrected': w_p
                    })

    w_df = pd.DataFrame(wilcox_rows)
    if w_df.empty:
        warnings.warn(f"No Wilcoxon pairs met n>={WILCOX_MIN_N} for {consensus}")
    else:
        if multipletests is not None:
            _, p_adj, _, _ = multipletests(w_df['wilcox_p_uncorrected'].values, alpha=ALPHA, method='bonferroni')
            w_df['wilcox_p_bonf'] = p_adj
            w_df['significant'] = p_adj < ALPHA
        w_df.to_csv(RESULTS_PATH / f'attribution_wilcoxon_{consensus}.csv', index=False)

    return agg_df



In [ ]:
print("Processing Primary Consensus 2of3")
df_2of3 = run_metrics_pipeline(PRIMARY_CONSENSUS)
agg_2of3 = aggregate_and_test(df_2of3, PRIMARY_CONSENSUS)



Processing Primary Consensus 2of3


Processing 2of3:   0%|          | 0/1528 [00:00<?, ?it/s]

/tmp/ipykernel_1904/3440203417.py:141: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc_miou = np.trapz(miou_vals, dx=0.01) / 0.99
/tmp/ipykernel_1904/3440203417.py:142: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc_miou_margin5 = np.trapz(miou_m5_vals, dx=0.01) / 0.99


Filter funnel (2of3): patho=1383 → TP=1030 → has_box=916 → has_ig=916 → output=916
✅ attribution_metrics_2of3.csv: 916 rows


In [ ]:
print("Processing Sensitivity Consensus 3of3")
df_3of3 = run_metrics_pipeline(SENS_CONSENSUS)
agg_3of3 = aggregate_and_test(df_3of3, SENS_CONSENSUS)



Processing Sensitivity Consensus 3of3


Processing 3of3:   0%|          | 0/1528 [00:00<?, ?it/s]

/tmp/ipykernel_1904/3440203417.py:141: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc_miou = np.trapz(miou_vals, dx=0.01) / 0.99
/tmp/ipykernel_1904/3440203417.py:142: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  auc_miou_margin5 = np.trapz(miou_m5_vals, dx=0.01) / 0.99


Filter funnel (3of3): patho=1383 → TP=1030 → has_box=737 → has_ig=737 → output=737
✅ attribution_metrics_3of3.csv: 737 rows


In [17]:
# Sanity Checks
print("\\n--- VALIDATION vs NB06 ---")
miou_old = pd.read_csv(RESULTS_PATH / 'miou_results.csv')
miou_old = miou_old[miou_old['subset'] == 'patho'].copy()
miou_old['image_id'] = miou_old['image_id'].astype(str)

tp_mask = miou_old.apply(
    lambda row: is_true_positive(row['image_id'], row['model'], row['target_class']),
    axis=1
)
miou_old_tp = miou_old[tp_mask]
print(f"miou_results patho rows: {len(miou_old)} | TP-filtered: {len(miou_old_tp)} | NB06.1 rows: {len(df_2of3)}")

cmp = df_2of3.merge(
    miou_old_tp[['image_id','model','target_class','miou']],
    on=['image_id','model','target_class'],
    how='inner'
)
assert len(cmp) == len(df_2of3), f"Row mismatch: cmp={len(cmp)} df={len(df_2of3)}"
match = np.allclose(cmp['miou_top50'], cmp['miou'], atol=1e-4, equal_nan=False)
print(f"miou_top50 matches miou_results.csv on TP rows? {match}")
if not match:
    diffs = np.abs(cmp['miou_top50'] - cmp['miou'])
    print(f"Max diff: {diffs.max():.6f} | n_mismatch: {(diffs > 1e-4).sum()}")

print("\\n--- SANITY CHECKS ---")
checks = []
checks.append(('TP rows > 100', len(df_2of3) > 100))
checks.append(('precision_mass > miou_top50', df_2of3['precision_mass'].mean() > df_2of3['miou_top50'].mean()))
checks.append(('mean delta_precision > 0', df_2of3['delta_precision'].mean() > 0))
# max_miou includes the top-50 point; mean AUC averages all thresholds (often lower)
checks.append(('max_miou >= miou_top50 (all rows)', (df_2of3['max_miou'] >= df_2of3['miou_top50'] - 1e-9).all()))
checks.append(('no torch imported', 'torch' not in sys.modules))
checks.append(('miou_top50 parity vs NB06', match))

for name, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")

critical = [name for name, ok in checks if not ok and name != 'miou_top50 parity vs NB06']
if critical:
    raise RuntimeError(f"Sanity checks failed: {critical}")
if not match:
    warnings.warn(
        "miou_top50 differs from NB06 miou_results.csv — results are still valid; "
        "re-run NB06 miou step if exact parity is required."
    )

print("\\n--- HEAD-TO-HEAD TABLE (Overall Mean) ---")
h2h = df_2of3.groupby('model')[['precision_mass', 'recall_top50', 'miou_top50', 'auc_miou', 'pointbiserial_r']].mean()
print(h2h)

print("\\nTo cite in paper, use attribution_faithfulness_2of3.csv")



\n--- VALIDATION vs NB06 ---
miou_results patho rows: 1008 | TP-filtered: 916 | NB06.1 rows: 916
miou_top50 matches miou_results.csv on TP rows? False
Max diff: 0.011214 | n_mismatch: 899
\n--- SANITY CHECKS ---
  [PASS] TP rows > 100
  [PASS] precision_mass > miou_top50
  [PASS] mean delta_precision > 0
  [PASS] max_miou >= miou_top50 (all rows)
  [PASS] no torch imported
  [FAIL] miou_top50 parity vs NB06
\n--- HEAD-TO-HEAD TABLE (Overall Mean) ---
                 precision_mass  recall_top50  miou_top50  auc_miou  \
model                                                                 
convnextv2_tiny        0.115521      0.305942    0.088464  0.073823   
densenet121            0.110017      0.244248    0.074650  0.063007   
swinb_lora             0.152284      0.591141    0.133419  0.115337   

                 pointbiserial_r  
model                             
convnextv2_tiny         0.107447  
densenet121             0.082770  
swinb_lora              0.277071  
\nTo cite in p

/tmp/ipykernel_1904/984504568.py:43: UserWarning: miou_top50 differs from NB06 miou_results.csv — results are still valid; re-run NB06 miou step if exact parity is required.
  warnings.warn(


In [ ]:
# [MARKDOWN]
# ## Methods for Paper
#
# **Why mIoU alone is insufficient**: Standard Mean Intersection-over-Union (mIoU) mathematically penalizes attribution maps for being highly focal. Because clinical bounding boxes annotate the full extent of a pathology rather than the minimal discriminative region required for prediction, an attribution map that perfectly highlights a discriminative focal point will score artificially low in classical mIoU.
#
# **Precision vs. Recall Interpretation**: To resolve this, we decouple mIoU into spatial Precision and Recall. Precision (Mass-in-Box) measures the fraction of the model's total continuous attribution mass that falls inside the ground truth box. Recall (Coverage) measures the fraction of the ground truth box area that receives top-50% binary attribution mass. Models that produce clinically focal, specific explanations are expected to exhibit high Precision and low Recall. `enrichment_ratio` (= precision_mass / box_area_fraction) is reported in supplementary materials only. It is unbounded and must NOT be labeled "recall" in the main text. Headline recall is `recall_top50`.
#
# **AUC-mIoU**: We compute Area Under the Curve for mIoU (AUC-mIoU) by sweeping the binarization mass threshold from top-1% to top-100% (in 1% increments), neutralizing any threshold-selection bias.
#
# **Point-Biserial Correlation**: To evaluate the models using 100% of the continuous attribution signal without any binarization, we compute the point-biserial correlation between the normalized, un-thresholded IG intensity array and the binary ground truth box mask.
#
# **5% Tolerance Margin**: To account for inherent looseness in radiologist bounding box annotations and CNN feature map discretization, we also evaluate mIoU against ground truth boxes expanded by a strict 5% tolerance margin (matching our False Positive spatial methodology).
#
# **Recommended Reporting**: In Table 2, we strongly recommend reporting `precision_mass`, `recall_top50`, and `auc_miou_margin5` as headline metrics, with legacy `miou_top50` moved to the supplement.
